# Module 4 — Grounding Agents in Telecom Knowledge (RAG)

**NetOps Co. · CELL-031A · ~45 minutes · needs a free Gemini API key**

Ask a model with no retrieval why CELL-031A keeps congesting and you get one of two things: a
generic checklist, or a confidently invented root cause. Neither is about NetOps Co.'s network,
because the model has never seen its incident history.

RAG fixes that — not by making the model smarter, but by putting the right past incident in front
of it before it answers. Five steps, and no vector-database internals required:

1. **Chunk** the incident write-ups
2. **Embed** every chunk
3. **Construct the query**: decide what you search with
4. **Retrieve** the most similar documents
5. **Augment** the prompt with them

Three things start a run like this, and they arrive with different material: an **alarm or KPI
breach** (telemetry, no words), a **ticket** (fields, plus how it was reported), or an **engineer
asking something** (words, maybe no telemetry). One ticket runs through this whole notebook,
**TCK-4471**:

> *"A trouble ticket (TCK-4471) reports slow data speeds near SITE-031 during evening peak hours
> for the past three days, with no specific alarm cited yet."*

It is a congestion problem. Watch which document the retriever thinks it is.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

**No key?** This notebook needs one for embeddings. Run `python module04-rag/rag_pipeline.py`
locally instead: it falls back to offline keyword matching.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os

# Repo root, resolved rather than hardcoded: the clone Colab just made, the
# checkout this notebook lives in, or the directory it was launched from.
REPO_ROOT = next((p for p in ('/content/netops-genai-course',
                              os.path.abspath(os.path.join(os.getcwd(), '..')),
                              os.getcwd())
                  if os.path.isdir(os.path.join(p, 'data'))), None)
assert REPO_ROOT, 'Could not find the repo root. Run this notebook from inside the checkout.'
sys.path.insert(0, os.path.join(REPO_ROOT, 'data'))

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working. These are also the live readings the ticket never mentions.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. Chunk — and what makes it "structure-aware"

We split each document on blank lines, so every section or paragraph becomes one chunk. A
runbook step sliced in half by a character count retrieves as nonsense.

Splitting alone creates a new problem: the paragraph that says *"customers reported slow
speeds"* no longer says which site it happened at. So every chunk gets its document's
**title and site** added on top. Look for that first line in square brackets below.

In [ ]:
sys.path.insert(0, os.path.join(REPO_ROOT, 'module04-rag'))
from rag_pipeline import (load_and_chunk_knowledge_base, retrieve, build_retrieval_query,
                          draft_grounded_rca, embed_with_gemini_api, _cosine_similarity)
import rag_pipeline
import nb_viz
from IPython.display import HTML, display

chunks = load_and_chunk_knowledge_base()
print(f'{len(chunks)} chunks from {len(set(c["source"] for c in chunks))} documents\n')

example = next(c for c in chunks if c['excerpt'].startswith('## Summary'))
print('WHAT GETS EMBEDDED:\n')
print(example['text'][:320], '…')

---
## 2 and 4. Embed, then retrieve with the ticket as written

We are skipping step 3 on purpose, and searching with the ticket text exactly as it arrived.
That is what most RAG code does.

Retrieval is **ranking, not lookup**. Every chunk is scored against the search text and sorted,
and each document is represented by its best chunk. There is no "nothing found": the failure
mode is **the wrong document won**.

**Before you run it, predict.** The knowledge base has four documents:

| Document | Site | About |
|---|---|---|
| `incident_001` | SITE-031 | evening congestion near an event venue |
| `incident_002` | SITE-031 | congestion from a neighbour cell's outage |
| `incident_003` | SITE-022 | VoLTE call drops, a different cell |
| `reference` | — | congestion and handover basics |

Which one ranks first for TCK-4471?

In [ ]:
QUESTION = ('A trouble ticket (TCK-4471) reports slow data speeds near SITE-031 during evening '
            'peak hours for the past three days, with no specific alarm cited yet.')

# Embed the knowledge base ONCE and reuse it. This model embeds one text per call,
# so this line is 25 calls (one rejected batch, then one per chunk); re-embedding
# per question would rate-limit a free key.
vectors = embed_with_gemini_api([c['text'] for c in chunks], task_type='RETRIEVAL_DOCUMENT')
# Hand the same vectors to retrieve(), which the pipeline uses in step 5.
rag_pipeline._chunk_vector_cache[hash(tuple(c['text'] for c in chunks))] = vectors

def rank_documents(search_text, top=4):
    """Every chunk scored against the search text; each document shown once, by its best chunk."""
    qv = embed_with_gemini_api([search_text], task_type='RETRIEVAL_QUERY')[0]
    scored = sorted(((c, _cosine_similarity(qv, v)) for c, v in zip(chunks, vectors)),
                    key=lambda p: -p[1])
    best = {}
    for c, score in scored:
        best.setdefault(c['source'], (c['source'], score, c['excerpt']))
    return list(best.values())[:top]

display(HTML(nb_viz.score_bars(rank_documents(QUESTION),
                               caption='Searching with the ticket as written')))

> **Reading the chart.** Bar lengths are relative to the top score, so read the numbers too.
> Scores only mean something compared with each other for the same search, and a small gap
> between first and second is not a verdict. It is a sign to check the top result yourself.

When this was measured, **`incident_003`, the VoLTE postmortem on a different
site, ranked first**. If you got the same, look at what the ticket actually says:

| Phrase in the ticket | Describes | Sounds most like |
|---|---|---|
| "a trouble ticket… reports" | how it was reported | `incident_003`: "tracked as a trouble ticket after repeated field reports" |
| "no specific alarm cited yet" | how it was reported | `incident_003`: "did not trigger a major or critical alarm" |
| "evening peak… past three days" | when it happens | `incident_003`: "tied to time-of-day traffic patterns" |
| "slow data speeds near SITE-031" | **the fault** | `incident_001`, the congestion postmortem |

The retriever was not broken. Embeddings encode *everything* in the text, including the
paperwork, and three of the four phrases are paperwork.

If you got a different winner, write down what you saw. A ranking is a measurement on a
particular model on a particular day, not a property of the code.

---
## 3. Construct the query — one ticket, two jobs

The ticket does two different jobs, and they need different text:

- **Search text → the retriever.** Built from what the network *measured*: alarm types, KPI
  thresholds crossed, cell and site IDs.
- **Question → the prompt.** The ticket as written, **unchanged**. Nothing is thrown away.

A senior engineer searching old tickets does the same thing: they type `PRB high CELL-031A`,
not the customer's complaint. `build_retrieval_query()` has three styles so you can compare:

| Style | Search text |
|---|---|
| `question` | the ticket as written (what you just ran) |
| `measured` | alarms + KPI breaches + IDs only |
| `measured+question` | measured facts first, then the ticket (the pipeline's default) |

In [ ]:
from mock_tools import get_cell_kpis, get_active_alarms, lookup_topology

CELL = 'CELL-031A'
SITE = lookup_topology(CELL)['site_id']
kpis, alarms = get_cell_kpis(CELL), get_active_alarms(SITE)

for style in ['question', 'measured', 'measured+question']:
    search_text = build_retrieval_query(QUESTION, kpis, alarms, CELL, SITE, style=style)
    ranked = rank_documents(search_text)
    print(f'{style:<18} 1st: {ranked[0][0]:<42} score {ranked[0][1]:.3f}')
    print(f'{"":<18} searched with: {search_text[:90]}{"…" if len(search_text) > 90 else ""}\n')

> The rule worth carrying to your own systems: **retrieve on what you measured and how it
> behaves, not on how it was reported.** Ticket numbers, who raised it, and whether an alarm
> was cited are routing metadata. They belong in the ticket, not in the search.

Why keep the question in the default at all? Because the data does not always show the problem.
A ticket about an aging radio unit has no alarm yet, and a search built only from alarms would
never find a hardware incident.

---
## 5. Augment — ungrounded vs grounded

The ungrounded call gets the ticket and nothing else. The grounded call is the pipeline's real
step 5, `draft_grounded_rca()`: the ticket, the live alarms and KPIs for CELL-031A, and the two
documents retrieved with the `measured+question` search text.

In [ ]:
from llm_client import call_llm

ungrounded = call_llm([{'role': 'user', 'content': QUESTION}])

trace = []
grounded = draft_grounded_rca(CELL, QUESTION, trace=trace)

print('── UNGROUNDED ' + '─' * 50)
print(ungrounded[:700])
print('\n── GROUNDED ' + '─' * 52)
print(grounded)          # printed in full: the Sources section at the end is the point
print('\nretrieved:', trace[0]['retrieved'])  # is the second document one you would have picked?

> **Read the grounded answer bottom-up.** It ends with a `Sources:` section naming the retrieved
> document it used and the one it discarded, and each recommended action is tagged either with the
> incident file it repeats or `[general practice]` — the model's own knowledge rather than NetOps
> Co.'s history. That is what makes a grounded answer checkable instead of merely confident.

> The ungrounded answer is not *worse written*. Depending on the run it is a generic checklist or
> a confident guess, and either way nothing in it comes from NetOps Co.'s history. Fluent is not
> the same as grounded. A wrong grounded answer is a better problem to have because you can
> inspect it: check the retrieved documents, the search text, and the ranking. One of the three
> will tell you what went wrong.

---
## Your turn

1. **Ask about something the knowledge base does not cover**, like a transmission link fault.
   Retrieval still returns its top documents. It never returns nothing. What do the scores look
   like compared with TCK-4471's?
2. **Describe symptoms two incidents share.** `incident_001` and `incident_002` both involve
   CELL-031A going over capacity. Which ranks first, how close is it, and what detail would
   actually tell them apart?
3. **See what bad retrieval does to the answer.** Draft the RCA with
   `query_style='question'` and compare it with the grounded one above. Did it pull in the
   VoLTE incident, and did the RCA follow it?
4. **Filter before you rank.** `k=2` returns two documents whether or not both are relevant.
   Every chunk carries its document's site, so one line keeps only this site's chunks:
   `pool = [c for c in chunks if 'SITE-031' in c['context']]`, then rank against `pool`.
   The VoLTE incident cannot win now. What else did that line drop, and when would that hurt?

In [ ]:
# 1. Not in the knowledge base
Q1 = 'SITE-014 transmission link flapped twice overnight and self-recovered. What caused it?'
display(HTML(nb_viz.score_bars(rank_documents(Q1), caption=Q1)))

# 2. Symptoms shared by two incidents
Q2 = 'CELL-031A active users jumped past planned capacity within minutes and RRC drops rose'
display(HTML(nb_viz.score_bars(rank_documents(Q2), caption=Q2)))

# 3. Uncomment to draft with the ticket as written (one LLM call)
# trace_q = []
# print(draft_grounded_rca(CELL, QUESTION, trace=trace_q, query_style='question')[:900])
# print('retrieved:', trace_q[0]['retrieved'])

---
**More practice:** run `python module04-rag/lab_query_construction.py` to compare all three query
styles side by side, then do **Checkpoint 1** in `checkpoints/01_rag/`: teach the retriever a new
incident that looks just like `incident_001` from the customer's side.

**Next:** everything so far is one prompt, one answer. Module 5 is where the model starts
deciding what to do next on its own.